#### 문항 1. 뒤죽박죽인 날짜 표기를 하나로 통일하기

여러 사이트에서 긁어온 날짜 문자열이 제각각이다. 이를 YYYY-MM-DD 하나로 통일하는 함수를 작성하시오.
```
samples = [
    "2024.12.24",          "2024-12-24",        "2024/12/24",
    "24.12.24",            "2024년 12월 24일",   "2024년 3월 5일",
    "12/24/2024",          "2024.12.24 14:30",  "등록일 : 2024.12.24",
    "2024-13-45",          "작성일 없음",         "",
]
```
#### 조건

* normalize_date(s) -> str | None 함수를 작성할 것

* 위 12가지를 모두 처리할 것 — 변환 불가능하면 None

* 두 자리 연도(24.12.24)는 2024로 해석할 것

* 월·일이 한 자리인 경우(3월 5일)도 03, 05로 채울 것

* 존재하지 않는 날짜(2024-13-45)는 None으로 처리할 것 — 정규표현식만으론 거를 수 없음. 후처리할 것

* 12/24/2024(미국식)와 2024/12/24를 구분할 것

* 각 입력에 대해 어떤 패턴으로 매치됐는지 함께 출력할 것 (기대결과 부분 확인)

※ 매칭된 것을 변수로써 꺼내는 방법 (명명 그룹)

(?P<변수명>정규표현식)
```
pattern = re.compile(r"(?P<loc>\d{3,4})\-(?P<mid>\d{3,4})\-(?P<last>\d{4})")
m = pattern.search('031-555-3331')
print(m['loc'])
# 031
```

기대 결과
```
2024.12.24            → 2024-12-24   [ymd_dot]
24.12.24              → 2024-12-24   [ymd_short]
2024년 3월 5일         → 2024-03-05   [ymd_kor]
12/24/2024            → 2024-12-24   [mdy_slash]
2024-13-45            → None         [ymd_dash · 유효하지 않은 날짜]
작성일 없음            → None         [매치 없음]
""                   → None         [빈 값]
```

In [ ]:
import re
from datetime import date

PATTERNS = [
    ("ymd_kor",   re.compile(r'(?<!\d)(?P<y>\d{4})년\s*(?P<m>\d{1,2})월\s*(?P<d>\d{1,2})일')),
    ("ymd_dot",   re.compile(r'(?<!\d)(?P<y>\d{4})\.(?P<m>\d{1,2})\.(?P<d>\d{1,2})(?!\d)')),
    ("ymd_dash",  re.compile(r'(?<!\d)(?P<y>\d{4})-(?P<m>\d{1,2})-(?P<d>\d{1,2})(?!\d)')),
    ("ymd_slash", re.compile(r'(?<!\d)(?P<y>\d{4})/(?P<m>\d{1,2})/(?P<d>\d{1,2})(?!\d)')),
    ("mdy_slash", re.compile(r'(?<!\d)(?P<m>\d{1,2})/(?P<d>\d{1,2})/(?P<y>\d{4})(?!\d)')),
    ("ymd_short", re.compile(r'(?<!\d)(?P<y>\d{2})\.(?P<m>\d{2})\.(?P<d>\d{2})(?!\d)')),
]

def normalize_date(s: str):
 
    if not s:
        return None, "빈 값"

    for name, pat in PATTERNS:
        m = pat.search(s)
        if not m:
            continue

        y, mo, d = int(m["y"]), int(m["m"]), int(m["d"])
        if y < 100:                
            y += 2000

        try:
            date(y, mo, d)                
        except ValueError:
            return None, f"{name} · 유효하지 않은 날짜"

        return f"{y:04d}-{mo:02d}-{d:02d}", name

    return None, "매치 없음"

samples = [
    "2024.12.24",          "2024-12-24",        "2024/12/24",
    "24.12.24",            "2024년 12월 24일",   "2024년 3월 5일",
    "12/24/2024",          "2024.12.24 14:30",  "등록일 : 2024.12.24",
    "2024-13-45",          "작성일 없음",         "",
]

for s in samples:
    result, tag = normalize_date(s)
    label = repr(s) if s == "" else s
    print(f"{label:<22}→ {str(result):<12} [{tag}]")

2024.12.24            → 2024-12-24   [ymd_dot]
2024-12-24            → 2024-12-24   [ymd_dash]
2024/12/24            → 2024-12-24   [ymd_slash]
24.12.24              → 2024-12-24   [ymd_short]
2024년 12월 24일         → 2024-12-24   [ymd_kor]
2024년 3월 5일           → 2024-03-05   [ymd_kor]
12/24/2024            → 2024-12-24   [mdy_slash]
2024.12.24 14:30      → 2024-12-24   [ymd_dot]
등록일 : 2024.12.24      → 2024-12-24   [ymd_dot]
2024-13-45            → None         [ymd_dash · 유효하지 않은 날짜]
작성일 없음                → None         [매치 없음]
''                    → None         [빈 값]


#### 문항 2. 서버 액세스 로그 파싱과 집계

웹 서버의 액세스 로그를 정규표현식으로 파싱해 분석하시오.

아래 내용을 담은 로그 파일 access.log 를 생성하시오.

203.0.113.42 - - [06/Aug/2026:14:22:31 +0900] "GET /list?page=3 HTTP/1.1" 200 5321 "<https://example.com/>" "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
198.51.100.7 - - [06/Aug/2026:14:22:33 +0900] "POST /api/search HTTP/1.1" 429 118 "-" "python-requests/2.31.0"
203.0.113.42 - - [06/Aug/2026:14:22:35 +0900] "GET /detail/9981 HTTP/1.1" 404 209 "<https://example.com/list>" "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
조건

하나의 정규표현식으로 한 줄에서 다음 7개를 추출할 것 
ip / timestamp / method / path / status / bytes / user_agent

명명 그룹 (?P<name>...) 을 사용할 것

형식이 깨진 줄은 건너뛰고, 몇 줄을 건너뛰었는지 출력할 것

다음 세 가지를 집계해 출력할 것

상태코드별 요청 수

4xx·5xx가 발생한 경로 상위 5개

봇으로 의심되는 User-Agent 목록과 그 요청 수 (bot / crawler / spider / python-requests 포함 여부로 판정, 대소문자 무시)

결과를 access_report.csv로 저장할 것

기대결과

파싱 3줄 · 건너뜀 0줄

── 상태코드별 요청 수 ──
status
200    1
404    1
429    1

── 4xx·5xx 발생 경로 상위 5 ──
path
/api/search     1
/detail/9981    1

── 봇 의심 User-Agent ──
user_agent
python-requests/2.31.0    1
  봇 요청 비율 33.3%

In [ ]:
import re
import csv
import pandas as pd

LOG_PATTERN = re.compile(
    r'^(?P<ip>\S+) \S+ \S+ '
    r'\[(?P<timestamp>[^\]]+)\] '
    r'"(?P<method>[A-Z]+) (?P<path>\S+) \S+" '
    r'(?P<status>\d{3}) (?P<bytes>\S+) '
    r'"(?P<referer>[^"]*)" '
    r'"(?P<user_agent>[^"]*)"$'
)

BOT_KEYWORDS = ("bot", "crawler", "spider", "python-requests")


def parse_log(path: str):
    rows, skipped = [], 0
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if not line.strip():
                continue
            m = LOG_PATTERN.match(line)
            if not m:
                skipped += 1
                continue
            rows.append(m.groupdict())
    return rows, skipped


def is_bot(ua: str) -> bool:
    ua_l = ua.lower()
    return any(k in ua_l for k in BOT_KEYWORDS)


def main():
    rows, skipped = parse_log(r"C:\Users\young\OneDrive\Desktop\campus\TIL\3_Crawl\access.log.txt")
    df = pd.DataFrame(rows)
    df["status"] = df["status"].astype(int)

    print(f"파싱 {len(df)}줄 · 건너뜀 {skipped}줄\n")

    status_counts = df["status"].value_counts().sort_index()
    print("── 상태코드별 요청 수 ──")
    print("status")
    for status, cnt in status_counts.items():
        print(f"{status}    {cnt}")
    print()

    err_df = df[df["status"] >= 400]
    path_counts = err_df["path"].value_counts().head(5)
    print("── 4xx·5xx 발생 경로 상위 5 ──")
    print("path")
    for path, cnt in path_counts.items():
        print(f"{path}     {cnt}")
    print()

    df["is_bot"] = df["user_agent"].apply(is_bot)
    bot_df = df[df["is_bot"]]
    bot_counts = bot_df["user_agent"].value_counts()
    print("── 봇 의심 User-Agent ──")
    print("user_agent")
    for ua, cnt in bot_counts.items():
        print(f"{ua}    {cnt}")
    bot_ratio = len(bot_df) / len(df) * 100
    print(f"  봇 요청 비율 {bot_ratio:.1f}%")

    out_cols = ["ip", "timestamp", "method", "path", "status", "bytes", "user_agent", "is_bot"]
    df[out_cols].to_csv(r"C:\Users\young\OneDrive\Desktop\campus\TIL\3_Crawl\access_report.csv", index=False, encoding="utf-8-sig")


if __name__ == "__main__":
    main()

파싱 3줄 · 건너뜀 0줄

── 상태코드별 요청 수 ──
status
200    1
404    1
429    1

── 4xx·5xx 발생 경로 상위 5 ──
path
/api/search     1
/detail/9981     1

── 봇 의심 User-Agent ──
user_agent
python-requests/2.31.0    1
  봇 요청 비율 33.3%


In [8]:

import os
folder = r"C:\Users\young\OneDrive\Desktop\campus\TIL\3_Crawl"
print(os.path.exists(folder))          # 폴더가 있는지
print(os.listdir(folder))              # 폴더 안에 뭐가 있는지

True
['260818_crawl.ipynb', '260819_crawl.ipynb', 'access.log.txt', 'aladin_bestseller.csv', 'hollys.csv']
